Install Dependencies

In [1]:
!pip install transformers flask flask-cors pyngrok peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjit

Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


Log in to Hugging Face

In [3]:
from huggingface_hub import login
login()


Load Model and Tokenizer

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# Path to your LoRA adapter in Google Drive
peft_path = "/content/drive/MyDrive/Colab Notebooks/WhatsApp Bot/gurt-mistral-lora-20250601-193305"

# Load base Mistral model from Hugging Face (requires login)
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    device_map="auto",
    torch_dtype=torch.float16
)

# Load tokenizer from the adapter directory (not base model)
tokenizer = AutoTokenizer.from_pretrained(peft_path)

# Load the fine-tuned LoRA adapter
model = PeftModel.from_pretrained(base_model, peft_path)

# Move to GPU or CPU
model.to("cuda" if torch.cuda.is_available() else "cpu")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

Set up Flask Server

In [5]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import regex as re
import time

app = Flask(__name__)
CORS(app)

def clean_output(text):
    text = re.sub(r'((\p{Emoji_Presentation}|\p{Extended_Pictographic})\uFE0F?\s?){2,}', lambda m: m.group(0).split()[0] + ' ', text)
    text = re.sub(r'([!?.,])\1{2,}', r'\1', text)
    text = re.sub(r'([!?.,]){2,}', lambda m: ''.join(set(m.group(0))), text)
    text = re.sub(r'\b(\w{2,})\b(?:\s+\1\b)+', r'\1', text, flags=re.IGNORECASE)
    text = re.sub(r'�{2,}', '', text)
    text = re.sub(r'[^\w\s.,!?🤔😐😂💀🔥☝️🗣️🔊]+$', '', text)
    return text.strip()

@app.route("/generate", methods=["POST"])
def generate():
    data = request.json
    prompt = data.get("prompt", "").strip() + "\nGurt:"

    if not prompt or len(prompt) < 10:
        return jsonify({"response": "Prompt too short."})

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    duration = round(time.time() - start_time, 2)
    print(f"Generated in {duration}s")

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Gurt:" in full_output:
        reply = full_output.split("Gurt:")[-1].strip().split("\n")[0]
    else:
        reply = full_output[len(prompt):].strip().split("\n")[0]

    reply = clean_output(reply)

    return jsonify({
        "response": reply,
        "generation_time": duration,
        "tokens_in": len(inputs['input_ids'][0]),
        "tokens_out": outputs.shape[-1]
    })


Authenticate Ngrok Authtoken

In [6]:
!ngrok config add-authtoken 2wvJt5WodUEHn0II3UzJL09WmwD_6cCoyQvPMqueu4uyuJyQq

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


Run with Ngrok

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(5000)
print("✅ Public URL:", public_url)

app.run(port=5000)


✅ Public URL: NgrokTunnel: "https://29dc-34-126-106-47.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [01/Jun/2025 22:16:58] "POST /generate HTTP/1.1" 200 -


⏱️ Generated in 4.63s


INFO:werkzeug:127.0.0.1 - - [01/Jun/2025 22:18:30] "POST /generate HTTP/1.1" 200 -


⏱️ Generated in 3.17s


...

⏱️ Generated in 3.16s


INFO:werkzeug:127.0.0.1 - - [01/Jun/2025 23:11:34] "POST /generate HTTP/1.1" 200 -


⏱️ Generated in 3.12s


INFO:werkzeug:127.0.0.1 - - [01/Jun/2025 23:12:02] "POST /generate HTTP/1.1" 200 -


⏱️ Generated in 3.14s
